# Nanopore base calling with Dorado

Jacobo de la Cuesta-Zuluaga. August 2026.

The aim of this notebook is to perform base calling from raw ONT `pod5` files: turning the raw signal
stored by the sequencer into DNA sequences in a `fastq` file. That file is the input of notebook 02.

This notebook covers a run with a single sample. Runs with several barcoded samples also need a
demultiplexing step, which is not covered here.

## Before we start

This notebook requires `conda` and the `VScode` environment of this repo. Instructions to install
conda are [here](https://conda.io/projects/conda/en/latest/user-guide/install/index.html); if you are
on the M3 cluster you should already have it.

The notebooks are written in **R**, not Python. In VSCode, click the kernel selector on the top right
and pick the R kernel from the `VScode` environment.

You'll also need access to a GPU partition on the cluster, and enough disk space: `pod5` files can
take up tens of GB. The `Nextflow` environment is only needed in notebook 02.

## What you'll need to change

These are the only values you have to edit. Everything else can stay as it is.

| Variable | Where | What to put there |
|---|---|---|
| `base_dir` | Load libraries and set paths | Directory where your input and output will live |
| `sample_names` | Test files | List of names of the samples to be processed |
| `raw_pod5_dirs` | Test files | List of folders that directly contains your `.pod5` files |
| `dorado_url` | Download base calling software | Only if you want a newer version of `dorado` |

## Load libraries and set paths

First, we'll set up the libraries and the work directory where we'll save our files.


In [301]:
# Libraries
library(tidyverse)
library(conflicted)

In [302]:
# Housekeeping: tells R which `filter` function to use when more than one
# package provides one. Nothing to change here.
conflicts_prefer(dplyr::filter())

[conflicted] Removing existing preference.
[conflicted] Will prefer dplyr::filter over any other package.


The following chunk will define the directories where the data is stored and where the output will be
saved. The present example assumes everything will be contained in the same directory: `base_dir`.
This might be different in your particular case, for example, if your sequences are stored on a
centralized directory or you have multiple runs stored in different folders. You can change this
accordingly.

`base_dir` has to exist already; the chunk only creates the folders inside it. Note down which one you
use, since notebook 02 needs the same one.

If a folder already exists, `dir.create` prints a warning. That's harmless.

In [ ]:
# Directories
# Base directory
# Change this to your own path
base_dir = "/PATH/TO/YOUR/FOLDER"

# Data
data_dir = file.path(base_dir, "data")
dir.create(data_dir)

# Sequences
# Where the fastq file produced by this notebook will end up
out_dir = file.path(data_dir, "fastq_files")
dir.create(out_dir)

# software dir
# Where dorado will be downloaded and uncompressed
bin_dir = file.path(base_dir, "bin")
dir.create(bin_dir)

# sheets dir
# Where the slurm script will be written
sheets_dir = file.path(data_dir, "sheets")
dir.create(sheets_dir)

Warning message:
In dir.create(data_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data' already exists
Warning message:
In dir.create(out_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/fastq_files' already exists
Warning message:
In dir.create(bin_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/bin' already exists
Warning message:
In dir.create(sheets_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/sheets' already exists


In [304]:
# Check that base was created
stopifnot(dir.exists(base_dir))

## Download base calling software

Next, we need to download `dorado`, which is the software we'll use to perform the base calling. It
comes as a compressed folder that we just unpack, so there's nothing to install.

To use a newer version, open the [dorado releases page](https://github.com/nanoporetech/dorado#installation), copy
the link of the latest `dorado-<version>-linux-x64.tar.gz`, and paste it below.

In the present example, we will use version 2.1.1.

The next two chunks unpack the download and store the location of the `dorado` program.

In [305]:
# Dorado file
# To update, replace this URL with the latest linux-x64 .tar.gz release
dorado_url <- "https://cdn.oxfordnanoportal.com/software/analysis/dorado-2.1.1-linux-x64.tar.gz"
dorado_destfile <- file.path(bin_dir, basename(dorado_url))

if (!file.exists(dorado_destfile)) {
  download.file(url = dorado_url, destfile = dorado_destfile, method = "wget")
  # Uncompress file
  ungz_cmd <- str_glue(
    "tar -zxf {dorado_destfile} -C {bin_dir}",
    dorado_destfile = dorado_destfile,
    bin_dir = bin_dir
  )
  system(ungz_cmd)
} else {
  print("File already exists")
}

[1] "File already exists"


In [ ]:
# Path to dorado executable
# Uncompressing creates a folder named like the download minus the .tar.gz
dorado_exec = file.path(str_remove(dorado_destfile, fixed(".tar.gz")), "bin/dorado")

# Check that executable works and register used version
system2(dorado_exec, "--version")

[2026-08-11 11:03:20.685] [info] Running: "--version"


2.1.1+d66c17c


## Test files

For the present example, we'll use publicly available sequencing data from
[Hall et al., 2024](https://doi.org/10.7554/eLife.98300.3). Specifically, this corresponds to the
sequencing of an isolate of _Streptococcus dysgalactiae_. 

The original file can be obtained [here](https://doi.org/10.26188/25495066.v1). On M3, a copy is
already available at the path below.

With your own data, mind that the sequencer produces nested folders and the `pod5` files sit a few
levels down, usually inside something like `pod5_skip/` or `pod5_pass/`.
Point to that folder, not to the top folder of the run.

For the present example, we will just use one sample, however, the code is set up to obtain
`fastq` files from `pod5` files of multiple runs. For this, we will create a table with the 
name of each sample and the folder where the `pod5` files of that sample run are located.

The name of the generated sequence files will be the same as the sample name. For example,
`SAMPLE_1` will be called `SAMPLE_1.fastq.gz`

In [307]:
# Sample names
sample_names = c("MMC234_202311")

# Folder that directly contains the .pod5 files, not the top folder of the run
# Note that this has to be the full path of the directory
raw_pod5_dirs = c("/mnt/lustre/groups/maier/databases/Huequito_Example/MMC234__202311")

In [308]:
# Create samples table
samples_table <- tibble(sample_names, raw_pod5_dirs) |>
  mutate(
    ArrayTaskID = row_number(),
    fastq = str_c(sample_names, ".fastq.gz"), 
    fastq = file.path(out_dir, fastq)
  ) |>
  relocate(ArrayTaskID)

# Show top of table
samples_table |>
  head()

# A tibble: 1 × 4
  ArrayTaskID sample_names  raw_pod5_dirs                                                      fastq                                                                                 
        <int> <chr>         <chr>                                                              <chr>                                                                                 
1           1 MMC234_202311 /mnt/lustre/groups/maier/databases/Huequito_Example/MMC234__202311 /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/fastq_…

In [309]:
# Write samples table to file
dorado_samplesfile = file.path(sheets_dir, "Example_dorado_samples.tsv")
write_tsv(samples_table,
    file = dorado_samplesfile)

The chunk below is the template of the script. Everything in square brackets is a placeholder that we
fill in afterwards, so there's nothing to edit here. The partition and the number of GPUs are specific
to M3; on another cluster those are the lines to adapt.

In [ ]:
# Template slurm file
# Do not modify this chunk. The values in square brackets are filled in below.
dorado_slurm_raw = str_glue(.open = "[", .close = "]",
"#!/bin/bash
##############################
#       Parameters           #
##############################

# This section tells the cluster what resources your job will need.
# These values are set in the notebook, in the chunk that fills this template.

# Name of the job
#SBATCH --job-name=[[job_name]]

# Generate an output file and give it a name
# This is the log of the run: check it if something goes wrong
#SBATCH --output=%x-%j.out

# Number of tasks
#SBATCH --ntasks=1

# Number of cpus that this task will need
#SBATCH --cpus-per-task=[[cpu]]

# Specify the total memory required per node
#SBATCH --mem=[[memory]]

# Specify the maximum time this job can take to run before being killed (hh:mm:ss)
#SBATCH --time=23:59:00

# Specify number of array jobs
#SBATCH --array=[[array_jobs]]

# Specify the partition to use
# This name is specific to M3
#SBATCH --partition=gpu-a30

# Type and number of gpus
#SBATCH --gres=gpu:2   
 
# Specify the path to the config file
samples_file=[[samples_file]]    

# Extract the sample name for the current $SLURM_ARRAY_TASK_ID
sample=$(awk -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $2}' $samples_file)

# Extract the folder with pod5 files for $SLURM_ARRAY_TASK_ID
pod5_dir=$(awk -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $3}' $samples_file)

# Extract the path of the output fastq for $SLURM_ARRAY_TASK_ID
out_fastq_gz=$(awk -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $4}' $samples_file)

# job information
scontrol show job ${SLURM_JOB_ID}
pwd

# per node
# do your real computation
source $HOME/.bashrc

# Set pipe fail
set -o pipefail

cd [[fastq_dir]]
[[dorado_exec]] basecaller [[accuracy_mode]] ${pod5_dir} --emit-fastq --trim [[trim_option]] --device cuda:all -v | gzip > ${out_fastq_gz}
")

Now we can replace the placeholders in the slurm script template with the actual paths and filenames
defined above. Two parameters are worth knowing:

- `accuracy_mode`: `hac` (high accuracy) is a good default. `sup` is more accurate but much slower,
  `fast` is only for a quick look.
- `trim_option`: `all` removes adapters and barcodes from the reads.

In [311]:
dorado_slurm = str_glue(dorado_slurm_raw, 
    job_name = "basecall_dorado",
    cpu = "16",
    memory = "128G",
    array_jobs = str_c("1-", nrow(samples_table)), # number of array jobs should be expressed as 1-<number of samples to run>, if 10 samples, 1-10
    samples_file = dorado_samplesfile,
    fastq_dir = out_dir,
    dorado_exec = dorado_exec,
    accuracy_mode = "hac",
    trim_option = "all",
    .open = "[", .close = "]")

dorado_slurm %>%
    print()

#!/bin/bash
##############################
#       Parameters           #
##############################

# This section tells the cluster what resources your job will need.
# These values are set in the notebook, in the chunk that fills this template.

# The success of your job depends on what is specified here.
# If you don't allocate enough resources (e.g. memory, cpus) your job will fail.
# If you allocate too much when not needed, your job will have a lower priority.

# The values used here are a sensible start.

# Set pipe fail
# set -o pipefail

# Name of the job
#SBATCH --job-name=basecall_dorado

# Generate an output file and give it a name
# This is the log of the run: check it if something goes wrong
#SBATCH --output=%x-%j.out

# Number of tasks
#SBATCH --ntasks=1

# Number of cpus that this task will need
#SBATCH --cpus-per-task=16

# Specify the total memory required per node
#SBATCH --mem=128G

# Specify the maximum time this job can take to run before being killed (hh:mm

The filled template can now be saved to a script to be submitted to the cluster using the `sbatch`
command.

In [312]:
# Write slurm file
dorado_slurmfile = file.path(sheets_dir, "dorado_slurm.sh")
write_lines(dorado_slurm, dorado_slurmfile)

The following chunk prints the full command for you to copy and run in your terminal.


In [313]:
# Execution command
str_glue("cd {data_dir} && sbatch {dorado_slurmfile}")

cd /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data && sbatch /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/sheets/dorado_slurm.sh

## While the job runs

The job runs on its own, so you can close the notebook and your terminal.

- Expect a few hours, longer in `sup` mode and depending on the sequencing depth.
- Check whether it's still running with `squeue`. If it's not listed, it's done.
- The log is the `.out` file in `data_dir`. Look there if something failed.

## What you should have at the end

The `fastq` files inside `data/fastq_files`. Each file should be at least a few
hundred MB. If it's only a few KB, there is an issue.

- It might be an issue with the sequencing run, check the sequencing logs.
- It could also be the base calling and likely failed. The `.out` file will say why.

As a quick sanity check, we can count the number of reads per sample in the following chunk

In [ ]:
# Obtain number of reads per sample
# Create a table
sequencing_depth = samples_table |> 
  pull(fastq) |> 
  map(function(fastq){
    n_reads = str_c("zcat", fastq, "| awk 'END {print NR/4}'", sep = " ") |> 
      system(intern = TRUE)
  tibble(sample = basename(fastq), n_reads = as.numeric(n_reads))
  }) |> 
  list_rbind()

In [318]:
# Print table
sequencing_depth


# A tibble: 1 × 2
  sample                 n_reads
  <chr>                    <dbl>
1 MMC234_202311.fastq.gz  236585

## Next

Continue with `02_Genome_assembly`, using the same `base_dir` as here. Wait until the base calling job
has finished before starting it.